# WebSocket Testing Notebook

This notebook tests the WebSocket functionality for the WasteWise application.

## Prerequisites
- Django server running on `localhost:8000`
- Redis server running for channel layers
- WebSocket endpoints accessible

## Installation
```bash
pip install websockets asyncio
```


In [2]:
import asyncio
import websockets
import json
import jwt
from datetime import datetime, timedelta
import time


In [3]:
# Configuration - Update these values
DJANGO_SECRET_KEY = 'your-secret-key'  # Replace with your Django SECRET_KEY
USER_ID = 1  # Replace with actual user ID
ADMIN_USER_ID = 1  # Replace with actual admin user ID
WS_BASE_URL = 'ws://localhost:8000'

print(f"🔧 Configuration:")
print(f"   Django Secret Key: {DJANGO_SECRET_KEY[:10]}...")
print(f"   User ID: {USER_ID}")
print(f"   Admin User ID: {ADMIN_USER_ID}")
print(f"   WebSocket URL: {WS_BASE_URL}")


🔧 Configuration:
   Django Secret Key: your-secre...
   User ID: 1
   Admin User ID: 1
   WebSocket URL: ws://localhost:8000


In [4]:
def generate_jwt_token(user_id, secret_key):
    """Generate a JWT token for WebSocket authentication"""
    payload = {
        'user_id': user_id,
        'exp': datetime.utcnow() + timedelta(hours=1)
    }
    return jwt.encode(payload, secret_key, algorithm='HS256')

# Generate tokens
user_token = generate_jwt_token(USER_ID, DJANGO_SECRET_KEY)
admin_token = generate_jwt_token(ADMIN_USER_ID, DJANGO_SECRET_KEY)

print(f"🔑 Generated Tokens:")
print(f"   User Token: {user_token[:50]}...")
print(f"   Admin Token: {admin_token[:50]}...")


🔑 Generated Tokens:
   User Token: eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJ1c2VyX2lkI...
   Admin Token: eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJ1c2VyX2lkI...


C:\Users\Ellis\AppData\Local\Temp\ipykernel_137404\3266561687.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'exp': datetime.utcnow() + timedelta(hours=1)


In [5]:
async def create_websocket_connection(endpoint, token):
    """Create a WebSocket connection with authentication"""
    uri = f"{WS_BASE_URL}{endpoint}?token={token}"
    
    try:
        websocket = await websockets.connect(uri)
        print(f"✅ Connected to {endpoint}")
        return websocket
    except Exception as e:
        print(f"❌ Failed to connect to {endpoint}: {e}")
        return None

async def send_message(websocket, message_type, data=None):
    """Send a message through WebSocket"""
    message = {
        "type": message_type,
        "data": data or {},
        "timestamp": datetime.utcnow().isoformat()
    }
    
    await websocket.send(json.dumps(message))
    print(f"📤 Sent: {message_type}")
    return message

async def receive_messages(websocket, timeout=5):
    """Receive messages from WebSocket with timeout"""
    messages = []
    
    try:
        while True:
            response = await asyncio.wait_for(websocket.recv(), timeout=timeout)
            message = json.loads(response)
            messages.append(message)
            print(f"📨 Received: {message.get('type', 'unknown')} - {message.get('message', '')}")
    except asyncio.TimeoutError:
        print(f"⏰ No more messages (timeout after {timeout}s)")
    except websockets.exceptions.ConnectionClosed:
        print("🔌 Connection closed")
    
    return messages


## Test 1: Customer WebSocket Connection

Test the general customer WebSocket endpoint:


In [6]:
async def test_customer_websocket():
    """Test customer WebSocket functionality"""
    print("🧪 Testing Customer WebSocket...")
    print("=" * 50)
    
    # Connect to customer WebSocket
    websocket = await create_websocket_connection("/ws/", user_token)
    if not websocket:
        return
    
    try:
        # Test authentication
        print("\n🔐 Testing Authentication...")
        await send_message(websocket, "auth", {"token": user_token})
        
        # Test ping/pong
        print("\n🏓 Testing Ping/Pong...")
        await send_message(websocket, "ping")
        
        # Test joining a chat room
        print("\n🏠 Testing Room Join...")
        await send_message(websocket, "join_room", {"roomId": "test-room-123"})
        
        # Test chat message
        print("\n💬 Testing Chat Message...")
        await send_message(websocket, "chat_message", {
            "roomId": "test-room-123",
            "message": "Hello from customer test!",
            "messageType": "text"
        })
        
        # Test typing indicator
        print("\n⌨️ Testing Typing Indicator...")
        await send_message(websocket, "chat_typing", {
            "roomId": "test-room-123",
            "isTyping": True
        })
        
        # Test read receipt
        print("\n👁️ Testing Read Receipt...")
        await send_message(websocket, "chat_read", {
            "roomId": "test-room-123",
            "messageId": "msg_test_123"
        })
        
        # Wait for responses
        print("\n⏳ Waiting for responses...")
        messages = await receive_messages(websocket, timeout=3)
        
        print(f"\n📊 Received {len(messages)} messages")
        
    finally:
        await websocket.close()
        print("\n🔌 Customer WebSocket connection closed")

# Run the test
await test_customer_websocket()


🧪 Testing Customer WebSocket...
✅ Connected to /ws/

🔐 Testing Authentication...
📤 Sent: auth

🏓 Testing Ping/Pong...
📤 Sent: ping

🏠 Testing Room Join...
📤 Sent: join_room

💬 Testing Chat Message...
📤 Sent: chat_message

⌨️ Testing Typing Indicator...
📤 Sent: chat_typing

👁️ Testing Read Receipt...
📤 Sent: chat_read

⏳ Waiting for responses...
📨 Received: connection_established - Connected to public real-time updates
📨 Received: auth_error - Authentication failed
📨 Received: pong - 
📨 Received: room_joined - 
📨 Received: error - Error processing message: 'AnonymousUser' object has no attribute 'email'
📨 Received: error - Error processing message: 'AnonymousUser' object has no attribute 'email'
📨 Received: chat_read - 


C:\Users\Ellis\AppData\Local\Temp\ipykernel_137404\1754189054.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat()


⏰ No more messages (timeout after 3s)

📊 Received 7 messages

🔌 Customer WebSocket connection closed
